49. Készítsünk listát a raktáron lévő termékek összértékéről raktárkód, azon belül kategóriakód, majd mennyiségi egység szerinti bontásban! A lista jelenítse meg a részösszegeket és a végösszeget is! A listát szűrjük az 5-ös és 9-es azonosítójú kategóriára! A csoportosításnál a ROLLUP záradékot használjuk!

In [ ]:
SELECT RAKTAR_KOD, 
        KAT_ID,
        MEGYS,
        SUM(LISTAAR*KESZLET) as 'Összérték'
from Termek
where KAT_ID in (5,9)
GROUP BY rollup (RAKTAR_KOD, KAT_ID, MEGYS)

50.Készítsünk listát a raktáron lévő termékek összértékéről raktárkód, azon belül kategóriakód, majd mennyiségi egység szerinti bontásban! A lista jelenítse meg a részösszegeket és a végösszeget is! A listát szűrjük az 5-ös és 9-es azonosítójú kategóriára! A csoportosításnál a CUBE záradékot használjuk!

In [ ]:
SELECT RAKTAR_KOD, 
        KAT_ID,
        MEGYS,
        SUM(LISTAAR*KESZLET) as 'Összérték'
from Termek
where KAT_ID in (5,9)
GROUP BY cube (RAKTAR_KOD, KAT_ID, MEGYS)

51. Készítsünk listát az egyes ügyfelek átlagos életkoráról az ügyfél neme, illetve az ügyfél születési éve szerint csoportosítva!A listát szűrjük azon ügyfelekre, akik neve D-vel vagy E-vel kezdődik! (Az életkor legyen a születési évtől a jelenlegi évig eltelt évek száma)

In [ ]:
SELECT szulev, nem,
AVG(YEAR(GETDATE())-SZULEV)
AS 'Átlagos életkor'
FROM Ugyfel
WHERE NEV LIKE 'E%' OR NEV LIKE 'D%' GROUP BY
GROUPING SETS((SZULEV),(NEM))

52. listázzuk, hogy melyik évben hány db terméket rendeltek meg! A lista megfelelően jelölve jelenítse meg a rendelések teljes összegét is!

In [1]:
SELECT
(
CASE GROUPING(YEAR(REND_DATUM))
WHEN 0 THEN CAST(YEAR(REND_DATUM)
AS nvarchar(4))
WHEN 1 THEN 'Összesen' END
)
AS ÉV,
COUNT(*) AS 'DB'
FROM Rendeles
GROUP BY ROLLUP(YEAR(REND_DATUM))

(4 rows affected)

Total execution time: 00:00:00.081

ÉV,DB
2015,9747
2016,14773
2017,3565
Összesen,28085


53. listázzuk, hogy naponta, azon belül fizetési mód szerint hány rendelés történt! A lista megfelelően jelölve jelenítse meg a részösszegeket és a végösszeget is!

In [ ]:
SELECT

IIF(
    GROUPING(REND_DATUM)=1,'Összesen',
CAST(REND_DATUM AS nvarchar(10))
) AS 'Rendelés dátuma',
CASE GROUPING_ID(REND_DATUM, FIZ_MOD)
    WHEN 0 THEN FIZ_MOD
    WHEN 1 THEN '**Fizetési módok összesen**'
    WHEN 3 THEN 'Összesen' 
END as 'FIZ_MOD',
COUNT(*) as 'DB'
FROM Rendeles
GROUP BY ROLLUP(REND_DATUM, FIZ_MOD)

54. <span style="background-color: rgb(255, 255, 255); color: rgb(0, 0, 0); font-family: &quot;Open Sans&quot;, sans-serif; font-size: 14.44px;">Készítsünk listát arról, hogy ügyfelenként (LOGIN), azon belül szállítási módonként hány megrendelés történt!&nbsp;</span>   

a. A lista tartalmazza a részösszegeket és a végösszeget is!  
b. Használjuk a ROLLUP záradékot!

In [1]:
SELECT [LOGIN],
        SZALL_MOD,
        COUNT(*)
from Rendeles
GROUP BY rollup ([LOGIN],SZALL_MOD)

(763 rows affected)

Total execution time: 00:00:00.071

LOGIN,SZALL_MOD,(No column name)
adam1,GLS,129
adam1,Posta,132
adam1,Személyes átvétel,131
adam1,NULL,392
adam3,GLS,10
adam3,Posta,3
adam3,Személyes átvétel,2
adam3,NULL,15
adam4,GLS,12
adam4,Posta,6


55. <span style="background-color: rgb(255, 255, 255); color: rgb(0, 0, 0); font-family: &quot;Open Sans&quot;, sans-serif; font-size: 14.44px;">Készítsünk listát a termékek számáról a következő csoportosítási szempontok szerint:</span>

<span style="background-color: rgb(255, 255, 255); color: rgb(0, 0, 0); font-family: &quot;Open Sans&quot;, sans-serif; font-size: 14.44px;">kategória azonosító, raktárkód, raktárkód+mennyiségi egység!</span>

a. A listát szűrjük azokra a csoportokra, ahol a termékek száma legalább 6!

In [2]:
SELECT KAT_ID,
        RAKTAR_KOD,
        MEGYS,
        COUNT(*) as 'db'
from Termek
GROUP by GROUPING SETS((KAT_ID),(RAKTAR_KOD),(RAKTAR_KOD,MEGYS))
HAVING COUNT(*)>5

(49 rows affected)

Total execution time: 00:00:00.026

KAT_ID,RAKTAR_KOD,MEGYS,db
NULL,1,db,11
NULL,1,NULL,11
NULL,3,db,14
NULL,3,NULL,14
NULL,5,csomag,7
NULL,5,db,72
NULL,5,NULL,86
NULL,6,csomag,9
NULL,6,db,94
NULL,6,NULL,106


```
56. Készítsünk listát az egyes termékkategóriákban lévő termékek számáról! 

```

a. Elég megjeleníteni a kategóriák azonosítóit és a darabszámokat!   
b. A lista megfelelően jelölve tartalmazza a végösszeget is!   
 c. Az oszlopokat nevezzük el értelemszerűen!   
 d. A listát rendezzük a darabszám szerint növekvő sorrendbe!

In [8]:
SELECT 
(
case GROUPING(KAT_ID)
    when 0 then cast(KAT_ID as nvarchar(50))
    when 1 then 'Összesen'
end 
) as 'kategória',
    COUNT(*) as 'db'
from Termek
group BY rollup(KAT_ID)
ORDER BY db ASC


(144 rows affected)

Total execution time: 00:00:00.036

kategória,db
10,1
11,1
12,1
15,1
17,1
26,1
29,1
31,1
33,1
34,1


57.  Készítsünk listát az ügyfelek számáról születési év szerint, azon belül nem szerinti bontásban!  
    
    a. A lista megfelelően jelölve tartalmazza a részösszegeket és a végösszeget is!  
    b. Az oszlopoknak adjunk nevet értelemszerűen!

In [9]:
SELECT 
    IIF(GROUPING(SZULEV)=1, 'Összesen', cast(SZULEV as nvarchar(20))) as 'Szülév',
    (
    case GROUPING_ID(SZULEV, NEM)
        when 0 then NEM
        when 1 then 'Nemek összesen'
        when 3 then 'Összesen'
    End 
    ) as 'Nemek',
    COUNT(*) as 'db'
from Ugyfel
GROUP by rollup(SZULEV, NEM)

(96 rows affected)

Total execution time: 00:00:00.016

Szülév,Nemek,db
1967,F,5
1967,N,3
1967,Nemek összesen,8
1968,F,5
1968,N,3
1968,Nemek összesen,8
1969,F,5
1969,N,3
1969,Nemek összesen,8
1970,F,4


58. <span style="background-color: rgb(255, 255, 255); color: rgb(0, 0, 0); font-family: &quot;Open Sans&quot;, sans-serif; font-size: 14.44px;">Készítsünk listát a termékek számáról a felvitel hónapja, azon belül napja szerint csoportosítva.&nbsp;</span>  

a. A lista csak a részösszegeket és a végösszeget tartalmazza!  
b. Az oszlopoknak adjunk megfelelő nevet!  
c. Ötlet: HAVING + GROUPING\_ID fv együttes használata

In [10]:
SELECT MONTH(felvitel) AS 'hónap',
       DAY(felvitel) AS 'nap',
       COUNT(*) AS 'termékek száma'
FROM Termek
GROUP BY ROLLUP(MONTH(felvitel), DAY(felvitel))
HAVING GROUPING_ID(MONTH(felvitel), DAY(felvitel)) > 0

(5 rows affected)

Total execution time: 00:00:00.014

hónap,nap,termékek száma
7,NULL,355
8,NULL,194
9,NULL,1
12,NULL,1
NULL,NULL,551


59. Jelenítsük meg a termékek kódja és listaára mellett a termékkategória átlagárát is!

In [11]:
select TERMEKKOD,
        LISTAAR,
        AVG(LISTAAR) OVER (PARTITION BY KAT_ID) as 'Kategória átlagár'
from Termek

(551 rows affected)

Total execution time: 00:00:00.031

TERMEKKOD,LISTAAR,Kategória átlagár
08070472T,1089,"621,1333333333333"
08070473T,1089,"621,1333333333333"
08070474T,1089,"621,1333333333333"
08070479T,674,"621,1333333333333"
08070480T,308,"621,1333333333333"
08070489T,868,"621,1333333333333"
08070490T,691,"621,1333333333333"
08070491T,214,"621,1333333333333"
08070492T,735,"621,1333333333333"
08070493T,720,"621,1333333333333"


60\. Listázzuk az egyes megrendelések dátumát, a termék kódját és mennyiségét, valamint a sorszám szerinti előző 5 megrendelés átlagos mennyiségét is!

In [14]:
SELECT r.REND_DATUM,
        rt.TERMEKKOD,
        rt.MENNYISEG,
        AVG(rt.MENNYISEG) OVER(PARTITION BY rt.TERMEKKOD
                                ORDER by rt.SORSZAM
                                rows BETWEEN 5 preceding and 1 preceding)
        as 'Előző 5 megrendelés átlagos mennyisége'
from Rendeles r JOIN Rendeles_tetel rt on r.SORSZAM=rt.SORSZAM

(163179 rows affected)

Displaying Top 5000 rows.

Total execution time: 00:00:02.472

REND_DATUM,TERMEKKOD,MENNYISEG,Előző 5 megrendelés átlagos mennyisége
2015-05-08,01010001T,20,NULL
2015-05-09,01010001T,80,20
2015-05-11,01010001T,90,50
2015-05-11,01010001T,30,"63,333333333333336"
2015-05-12,01010001T,40,55
2015-05-15,01010001T,10,52
2015-05-17,01010001T,20,50
2015-05-18,01010001T,30,38
2015-05-23,01010001T,20,26
2015-05-27,01010001T,70,24


61. Jelenítsük meg, hogy az egyes ügyfelek az adott rendelési dátumig bezárólag összesen hányszor rendeltek! Megjelenítendő a rendelés dátuma, az ügyfél login-ja és a rendelés darabszáma

In [16]:
SELECT distinct REND_DATUM, 
        [LOGIN],
        COUNT(*) OVER(PARTITION by LOGIN
                        ORDER BY REND_DATUM
                        range BETWEEN unbounded preceding and current row)
        as 'Eddigi rendelések'
from Rendeles

(21154 rows affected)

Displaying Top 5000 rows.

Total execution time: 00:00:00.597

REND_DATUM,LOGIN,Eddigi rendelések
2015-05-01,ANDRASE,1
2015-05-01,andrea4,1
2015-05-01,aniko4,1
2015-05-01,ATTILAO,1
2015-05-01,balazs2,1
2015-05-01,balint1,1
2015-05-01,BELAF,2
2015-05-01,GABORS,1
2015-05-01,gusztav,1
2015-05-01,JOZSEFG,1


62\. Készítsünk sorszámozott listát nemenként az ügyfelekről! A sorszámozás szempontja az ügyfél email-címe legyen!

In [19]:
SELECT ROW_NUMBER() OVER(PARTITION by NEM
                        ORDER BY EMAIL) 
        as 'nemenkénti sorszám',
        *
from Ugyfel

(200 rows affected)

Total execution time: 00:00:00.073

nemenkénti sorszám,LOGIN,EMAIL,NEV,SZULEV,NEM,CIM
1,adam4,ádám.bieniek@mail.hu,Bieniek Ádám,1976,F,"8630 Balatonboglár, Juhászföldi út 1."
2,adam1,ádám.kiss@mail.hu,Kiss Ádám,1991,F,"5630 Békés, Szolnoki út 8."
3,adam3,adam3@gmail.com,Barkóci Ádám,1970,F,"3910 Tokaj, Dózsa György utca 37."
4,akos,ákos.bíró@mail.hu,Bíró Ákos,1982,F,"9023 Győr, Kossuth Lajos utca 47/b."
5,aladar,aladár.dunai@mail.hu,Dunai Aladár,1980,F,"5931 Nagyszénás, Árpád utca 23."
6,alexis,alexbiro@gmail.com,Biró Alexander,2000,F,"6914 Pitvaros, Deák F. u. 38."
7,andras21,andrás.molnár@mail.hu,Molnár András,1977,F,"7900 Szigetvár, Rákóczi utca 67."
8,ANDRASN,andrás.nagy@mail.hu,Nagy András,1980,F,"6500 Baja, Fő út 169."
9,andras2,andrás.tóth@mail.hu,Tóth András,1997,F,"4071 Egyek, Petőfi utca 30."
10,andras3,andrás.vígh@mail.hu,Vígh András,1971,F,"1118 Budapest, Arany János utca 1."


63. Listázzuk a termékek kódját, megnevezését, kategória kódját, készlet mennyiségét és azt, hogy a termék a készlet alapján hányadik a kategóriájában

In [20]:
SELECT TERMEKKOD,
        MEGNEVEZES,
        KAT_ID,
        KESZLET,
        RANK() OVER(PARTITION BY KAT_ID
                    ORDER BY KESZLET desc)
        as 'Készlet szerinti helyezés'
from Termek

(551 rows affected)

Total execution time: 00:00:00.032

TERMEKKOD,MEGNEVEZES,KAT_ID,KESZLET,Készlet szerinti helyezés
08070480T,Fizika 13 éveseknek,4,5000,1
08070489T,RAMba zárt világ,4,4400,2
08070491T,Szám.tech. kicsiknek,4,1300,3
08070494T,Érettségi felv. fel. Fizika,4,400,4
08070479T,Fizika,4,200,5
08070485T,Jól felkészültem-e - Fizika,4,200,5
08070473T,A föld amelyen élünk - Távoli földrészek,4,200,5
08070474T,A föld amelyen élünk - Hazánk földrajza,4,100,8
08070472T,A föld amelyen élünk - Európa földrajza,4,100,8
08070483T,Fizikai feladatok és tévedések,4,100,8


64. Az előző példa DENSE\_RANK() függvénnyel

In [21]:
SELECT TERMEKKOD,
        MEGNEVEZES,
        KAT_ID,
        KESZLET,
        DENSE_RANK() OVER(PARTITION BY KAT_ID
                    ORDER BY KESZLET desc)
        as 'Készlet szerinti helyezés'
from Termek

(551 rows affected)

Total execution time: 00:00:00.087

TERMEKKOD,MEGNEVEZES,KAT_ID,KESZLET,Készlet szerinti helyezés
08070480T,Fizika 13 éveseknek,4,5000,1
08070489T,RAMba zárt világ,4,4400,2
08070491T,Szám.tech. kicsiknek,4,1300,3
08070494T,Érettségi felv. fel. Fizika,4,400,4
08070479T,Fizika,4,200,5
08070485T,Jól felkészültem-e - Fizika,4,200,5
08070473T,A föld amelyen élünk - Távoli földrészek,4,200,5
08070474T,A föld amelyen élünk - Hazánk földrajza,4,100,6
08070472T,A föld amelyen élünk - Európa földrajza,4,100,6
08070483T,Fizikai feladatok és tévedések,4,100,6


65. Listázzuk minden rendelési tétel sorszámát, a termék kódját és mennyiségét, valamint az adott termék előző rendelésének mennyiségét!

In [ ]:
SELECT SORSZAM, TERMEKKOD, MENNYISEG,
LAG(MENNYISEG,1,0) OVER(PARTITION BY
TERMEKKOD ORDER BY SORSZAM)
AS 'Előző rendelési mennyiség'
FROM Rendeles_tetel

66. Listázzuk minden rendelési tétel sorszámát, a termék kódját és mennyiségét, valamint az adott termék kettővel későbbi rendelésének mennyiségét!

In [22]:
SELECT SORSZAM,
        TERMEKKOD,
        MENNYISEG,
        LEAD(MENNYISEG,2,0) OVER(PARTITION BY TERMEKKOD
                                ORDER BY SORSZAM)
from Rendeles_tetel

(163179 rows affected)

Displaying Top 5000 rows.

Total execution time: 00:00:02.267

SORSZAM,TERMEKKOD,MENNYISEG,(No column name)
298,01010001T,20,90
321,01010001T,80,30
400,01010001T,90,40
416,01010001T,30,10
438,01010001T,40,20
587,01010001T,10,30
654,01010001T,20,20
702,01010001T,30,70
869,01010001T,20,60
1032,01010001T,70,60


67.  Listázzuk az egyes ügyfelek adatait és első rendelésük dátumát! A lista ne tartalmazzon duplikált sorokat!

In [23]:
SELECT distinct u.*,
                FIRST_VALUE(r.REND_DATUM)
                OVER(PARTITION BY u.LOGIN
                    ORDER by r.REND_DATUM)
                as 'első rendelés'
from Ugyfel u JOIN Rendeles r on u.[LOGIN]=r.[LOGIN]

(191 rows affected)

Total execution time: 00:00:00.448

LOGIN,EMAIL,NEV,SZULEV,NEM,CIM,(No column name)
adam1,ádám.kiss@mail.hu,Kiss Ádám,1991,F,"5630 Békés, Szolnoki út 8.",2015-05-04
adam3,adam3@gmail.com,Barkóci Ádám,1970,F,"3910 Tokaj, Dózsa György utca 37.",2015-05-04
adam4,ádám.bieniek@mail.hu,Bieniek Ádám,1976,F,"8630 Balatonboglár, Juhászföldi út 1.",2015-05-02
agnes,agnes@gmail.com,Lengyel Ágnes,1979,N,"5200 Törökszentmiklós, Deák Ferenc út 5.",2015-05-05
agnes3,agnes3@gmail.com,Hartyánszky Ágnes,1967,N,"6430 Bácsalmás, Posta köz 2.",2015-05-05
AGNESH,AGNESH@gmail.com,Horváth Ágnes,1981,N,"8200 Veszprém, Rákóczi utca 21.",2015-05-08
AGNESK,AGNESK@gmail.com,Kovács Ágnes,1988,N,"1084 Budapest, Endrődi Sándor utca 47.",2015-05-05
akos,ákos.bíró@mail.hu,Bíró Ákos,1982,F,"9023 Győr, Kossuth Lajos utca 47/b.",2015-05-16
aladar,aladár.dunai@mail.hu,Dunai Aladár,1980,F,"5931 Nagyszénás, Árpád utca 23.",2015-05-04
alexandra,alexandra.bagóczki@mail.hu,Bagóczki Alexandra,1992,N,"2381 Táborfalva, Petőfi utca 1/2.",2015-05-06


68. Soroljuk be a termékeket kategóriájukban a listaáruk alapján 5 osztályba!

In [24]:
SELECT *,
        ntile(5) OVER(PARTITION BY KAT_ID
                    order by LISTAAR)
        as 'Osztály'
from termek

(551 rows affected)

Total execution time: 00:00:00.025

TERMEKKOD,MEGNEVEZES,KAT_ID,LISTAAR,LEIRAS,RAKTAR_KOD,KESZLET,MEGYS,FELVITTE,FELVITEL,(No column name)
08070491T,Szám.tech. kicsiknek,4,214,NULL,8,1300,db,Sára,2016-08-20,1
08070485T,Jól felkészültem-e - Fizika,4,291,NULL,6,200,db,Béla,2016-08-22,1
08070480T,Fizika 13 éveseknek,4,308,NULL,8,5000,db,Béla,2016-08-29,1
08070483T,Fizikai feladatok és tévedések,4,324,NULL,5,100,db,Béla,2016-08-25,2
08070484T,Fogalmazás lépésről lépésre,4,345,NULL,8,100,db,Béla,2016-08-24,2
08070494T,Érettségi felv. fel. Fizika,4,440,NULL,6,400,db,Sára,2016-08-20,2
08070495T,Érettségi felv. fel. Biológia,4,440,NULL,9,100,db,Sára,2016-08-18,3
08070479T,Fizika,4,674,NULL,7,200,db,Béla,2016-08-29,3
08070490T,Szövegszerkesztés,4,691,NULL,8,100,db,Sára,2016-08-20,3
08070493T,Érettségi felv. fel. Matematika,4,720,NULL,5,100,db,Sára,2016-08-20,4


69. <span style="color: rgb(36, 41, 47); font-family: -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, Helvetica, Arial, sans-serif, &quot;Apple Color Emoji&quot;, &quot;Segoe UI Emoji&quot;; font-size: 16px;">Készítsünk listát éves bontásban norbert2 azonosítójú ügyfél rendeléseinek értékéről!</span>
    
    1. A lista megfelelően jelölve tartalmazza a végösszeget is!

In [27]:
SELECT
(
    case GROUPING(YEAR(r.rend_datum))
        when 0 then CAST(YEAR(r.rend_datum) as nvarchar(4))
        when 1 then 'Összesen'
    END
) as 'Év',
sum(rt.egysegar*rt.mennyiseg)
from Ugyfel u JOIN Rendeles r on u.LOGIN=r.LOGIN
                JOIN Rendeles_tetel rt on r.sorszam=rt.sorszam
WHERE r.LOGIN='norbert2'
group BY ROLLUP(YEAR(r.rend_datum))

(3 rows affected)

Total execution time: 00:00:00.022

Év,(No column name)
2015,483473
2017,614340
Összesen,1097813


70.  Készítsünk listát szállítási dátumonként, azon belül szállítási módonként az egyes rendelések összmennyiségéről!
    1. Csak azokat a termékeket vegyük figyelembe, amelyek mennyiségi egysége db!
    2. A listát szűrjük úgy, hogy az csak a részösszeg sorokat és a végösszeget tartalmazza!

In [33]:
SELECT IIF(GROUPING(r.szall_datum)=1, 'Összesen', 
            CAST(r.szall_datum as NVARCHAR(10))) as 'Dátum',
        (
            case GROUPING_ID(r.SZALL_DATUM, r.SZALL_MOD)
                when 0 then r.SZALL_MOD
                when 1 then 'Szállítási módok összesen'
                when 3 then 'Összesen'
            End
        ) as 'Szall_mod',
        sum(rt.mennyiseg) as 'Összmennyiség'
FROM Rendeles r JOIN Rendeles_tetel rt on r.sorszam=rt.sorszam
                join termek t on rt.TERMEKKOD=t.TERMEKKOD
where t.MEGYS='db'
group by ROLLUP(r.szall_datum, r.Szall_mod)
HAVING GROUPING_ID(r.SZALL_DATUM, r.SZALL_MOD) in (1,3)

(711 rows affected)

Total execution time: 00:00:01.015

Dátum,Szall_mod,Összmennyiség
2015-05-03,Szállítási módok összesen,100
2015-05-04,Szállítási módok összesen,996
2015-05-05,Szállítási módok összesen,3152
2015-05-06,Szállítási módok összesen,5030
2015-05-07,Szállítási módok összesen,3828
2015-05-08,Szállítási módok összesen,3213
2015-05-09,Szállítási módok összesen,6665
2015-05-10,Szállítási módok összesen,6909
2015-05-11,Szállítási módok összesen,6224
2015-05-12,Szállítási módok összesen,7568


In [30]:
SELECT r.SZALL_DATUM, 
       r.SZALL_MOD, 
       SUM(rt.MENNYISEG) AS 'Összmennyiség'
FROM Rendeles_tetel rt JOIN Termek t ON rt.TERMEKKOD = t.TERMEKKOD
                       JOIN Rendeles r ON r.SORSZAM = rt.SORSZAM
WHERE t.MEGYS='db'
GROUP BY ROLLUP(r.SZALL_DATUM, r.SZALL_MOD)
HAVING GROUPING_ID(r.SZALL_DATUM, r.SZALL_MOD) IN (1,3)

(711 rows affected)

Total execution time: 00:00:01.096

SZALL_DATUM,SZALL_MOD,Összmennyiség
2015-05-03,NULL,100
2015-05-04,NULL,996
2015-05-05,NULL,3152
2015-05-06,NULL,5030
2015-05-07,NULL,3828
2015-05-08,NULL,3213
2015-05-09,NULL,6665
2015-05-10,NULL,6909
2015-05-11,NULL,6224
2015-05-12,NULL,7568


```
71. Hány olyan ügyfél van, aki még nem rendelt semmit?

```

1. Csoportosítsuk őket nem szerint, azon belül életkor szerint!
2. A lista tartalmazza a részösszegeket és a végösszeget is!

In [38]:
SELECT u.nem,
        YEAR(GETDATE()) - u.SZULEV as 'Életkor',
        COUNT(*)
from ugyfel u left JOIN rendeles r on u.LOGIN=r.LOGIN
where r.LOGIN is NULL
GROUP BY ROLLUP( u.nem,  YEAR(GETDATE()) - u.SZULEV)

(12 rows affected)

Total execution time: 00:00:00.032

nem,Életkor,(No column name)
F,25,1
F,27,1
F,29,1
F,46,1
F,56,1
F,57,1
F,NULL,6
N,31,1
N,38,1
N,50,1


72. <span style="color: rgb(36, 41, 47); font-family: -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, Helvetica, Arial, sans-serif, &quot;Apple Color Emoji&quot;, &quot;Segoe UI Emoji&quot;; font-size: 16px;">Készítsünk listát a megrendelt termékek legkisebb és legnagyobb egységáráról szállítási dátum, azon belül szállítási mód szerinti bontásban!</span>

1. A lista csak a 2015 májusi szállításokat tartalmazza!
2. Jelenítsük meg a részösszegeket és a végösszeget is!

In [39]:
SELECT r.szall_datum,  
        r.Szall_mod,
        min(rt.egysegar) as 'legkisebb',
        MAX(rt.egysegar) as 'legnagyobb'
from Rendeles_tetel rt left JOIN rendeles r on rt.SORSZAM=r.sorszam
where YEAR(r.szall_datum)=2015 AND MONTH(r.szall_datum)=5
GROUP BY ROLLUP( r.szall_datum,  r.Szall_mod)

(115 rows affected)

Total execution time: 00:00:00.094

szall_datum,Szall_mod,legkisebb,legnagyobb
2015-05-03,GLS,17,400
2015-05-03,NULL,17,400
2015-05-04,GLS,9,1251
2015-05-04,Posta,12,15363
2015-05-04,Személyes átvétel,23,15363
2015-05-04,NULL,9,15363
2015-05-05,GLS,18,29090
2015-05-05,Posta,10,29090
2015-05-05,Személyes átvétel,11,366
2015-05-05,NULL,10,29090


zhgyak 1.  

Kérdezzük le, hogy melyik ügyfél (USERNEV) hány különbözö szálláshelyen foglalt!

a. A listában azok az ügyfelek is jelenjenek meg, akiknek még nem volt foglalásuk

b. Megfelelően jelölve jelenjen meg a végösszeg is!

In [6]:
SELECT
(
    case GROUPING(v.usernev)
    when 0 then v.USERNEV
    when 1 then 'Összesen'
    end 
) as 'Ügyfél',
COUNT(distinct sz.SZALLAS_FK) as 'db'
from Vendeg v left join Foglalas f on v.usernev=f.UGYFEL_FK
                    JOIN Szoba sz on f.SZOBA_FK=sz.SZOBA_ID
GROUP BY rollup(v.usernev)

(198 rows affected)

Total execution time: 00:00:00.028

Ügyfél,db
,3
adam1,4
adam3,2
adam4,6
agnes,4
agnes3,4
AGNESH,6
AGNESK,3
akos,9
aladar,4


zhgyak 2.

Készítsünk listát, amely megjeleníti a vendégek adatait!

· Egy új oszlopban számoljuk ki a vendég életkorát (években)

· Egy másik új oszlopban határozzuk meg, hogy születési dátum szerint növekvö rendezésnél mennyi az adott ügyfél. az előtte lévö 2 ügyfél és az utána lévő 2 ügyfél átlagos életkora! Az oszlopot

nevezzük el értelemszerűen!

In [12]:
SELECT *,
       YEAR(GETDATE())-YEAR(szul_dat) as 'életkor',
        AVG( YEAR(GETDATE())-YEAR(szul_dat)) 
                OVER(
                    ORDER by SZUL_DAT  
                    rows BETWEEN 2 preceding and 2 following)
from Vendeg
ORDER by SZUL_DAT asc

Warning: Null value is eliminated by an aggregate or other SET operation.

(213 rows affected)

Total execution time: 00:00:00.020

USERNEV,NEV,EMAIL,SZAML_CIM,SZUL_DAT,életkor,(No column name)
,Kiss József,kissjozsef@vhol.com,NULL,NULL,NULL,NULL
haliho,Nagy Árpád,nagyarpi@gmail.com,-,NULL,NULL,58
halloka,Kiss Benedek,kisbeni@gmli.com,,NULL,NULL,58
anett3,Pivarcsi Anett,anett.pivarcsi@mail.hu,1149 Budapest Fő út 60.,1967-01-03,58,58
gusztav,Bárci Gusztáv,gusztav@gmail.com,3643 Dédestapolcsány Endrődi Sándor utca 47.,1967-02-25,58,58
agnes3,Hartyánszky Ágnes,agnes3@gmail.com,6430 Bácsalmás Posta köz 2.,1967-04-11,58,58
eva,Enyedi Éva,eva@gmail.com,4231 Bököny Petőfi utca 8.,1967-06-30,58,58
ROBERTI,Iván Róbert,róbert.iván@mail.hu,2377 Örkény Petőfi Sándor utca 3.,1967-07-02,58,58
SZILARDS,Szalai Szilárd,szilárd.szalai@mail.hu,1077 Budapest Fő út 18.,1967-09-28,58,58
laszlo1,Farkas László,lászló.farkas@mail.hu,5200 Törökszentmiklós Rendeki utca 21.,1967-10-30,58,57


Egészítsük ki a megkezdett lekérdezést, amely listázza azon vendégek azonosítóját és nevét, akik már legalább egyszer foglaltak, és MINDEN ESETBEN összesen két fő számára (felnőtt + gyermek

szám összege)! a. A lista ne tartalmazzon ismetlodo sorokat!

In [40]:
SELECT distinct v.USERNEV,
        v.NEV
FROM Vendeg v JOIN Foglalas f ON v.USERNEV = f.UGYFEL_FK
WHERE NOT EXISTS
(
    SELECT 1
    FROM Foglalas f2
    WHERE f2.UGYFEL_FK = v.USERNEV 
        AND f2.FELNOTT_SZAM +f2.GYERMEK_SZAM <>2
)

(11 rows affected)

Total execution time: 00:00:00.031

USERNEV,NEV
sandor,Karasz Sándor
andras4,Back András
ANDRASN,Nagy András
szabolcs,Bodor Szabolcs
Barnabsds,
jozsef,Gergely József
kristof4,Poprádi Kristóf
tibor2,Dániel Tibor
balu,Endresz Bálint
timea,Papós Tímea
